# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Nome 1 (RM)
- Nome 2 (RM)
- Nome 3 (RM)

**Tema escolhido:** (escreva aqui o tema da lista do README)

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [ ]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [ ]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

In [ ]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """Você é um assistente de casa inteligente.
Ajude o usuário com dúvidas sobre automação residencial, sensores e dispositivos conectados.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com casa inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [ ]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [ ]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista": SYSTEM_PROMPT,
    "Professor para crianças": "Você explica tecnologia para crianças de 10 anos, usando comparações do dia a dia. Responda em português.",
    "Resumido": "Você responde qualquer pergunta em uma única frase curta, em português.",
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Como um sensor de presença pode ajudar a economizar energia em casa?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system))
    print()

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- Qual `system` gerou a resposta mais útil para o tema? Por quê?

(responda aqui)

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [ ]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [ ]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Quais sensores são mais usados em uma casa inteligente?",
    "Qual a diferença entre Wi-Fi e Zigbee para dispositivos da casa?",
    "Quais cuidados de segurança devo ter com câmeras conectadas?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))
    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))
    print("\n" + "=" * 60 + "\n")

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?

(responda aqui)

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [ ]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

In [ ]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Explique, com suas palavras, por que isso acontece.

(responda aqui)

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [ ]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [ ]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="Assistente de Casa Inteligente",
    description="Pergunte sobre automação residencial, sensores e dispositivos conectados.",
    examples=[
        ["Como funciona uma lâmpada inteligente?", "Hugging Face"],
        ["Vale a pena usar tomadas conectadas?", "Gemini"],
    ],
).launch(share=True)

**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [ ]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

In [ ]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

In [ ]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é IoT?"})
print(r.status_code)
print(r.json()["resposta"])

In [ ]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")